# Démo — Classification d'ECG (MIT-BIH, classes AAMI)

Pré-requis : `python scripts/download_data.py`, puis un entraînement, par exemple `python -m src.train --model resnet --augment`.
Ce notebook couvre : (1) tableau descriptif, (2) signaux par classe, (3) augmentation, (4) prédiction avec explication (Grad-CAM, attention).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from src.augmentation import jitter, magnitude_warp, time_warp
from src.data_processing import CLASS_DESCR, CLASS_NAMES, FS, WINDOW, describe_dataset, load_dataset, split_indices
from src.interpretability import GradCAM1D, attention_weights, plot_attention, plot_gradcam
from src.models import load_checkpoint
from src.utils import set_seed

set_seed(42)
data = load_dataset(ROOT / "data")
splits = split_indices(data)
print(f"fréquence d'échantillonnage : {FS} Hz | segment : {WINDOW} échantillons = {WINDOW / FS:.2f} s")
describe_dataset(data, splits)

## Signaux typiques par classe

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(len(CLASS_NAMES), 3, figsize=(12, 9), sharex=True)
for c, name in enumerate(CLASS_NAMES):
    idx = np.where(data["y"] == c)[0]
    for ax, i in zip(axes[c], rng.choice(idx, size=min(3, len(idx)), replace=False)):
        ax.plot(np.arange(WINDOW) / FS, data["X"][i], color="black", lw=1)
    axes[c, 0].set_ylabel(name)
axes[-1, 1].set_xlabel("temps (s)")
plt.tight_layout()

## Spectrogramme moyen par classe

In [ ]:
from scipy.signal import spectrogram

fig, axes = plt.subplots(1, len(CLASS_NAMES), figsize=(14, 2.8), sharey=True)
for c, (ax, name) in enumerate(zip(axes, CLASS_NAMES)):
    idx = np.where(data["y"] == c)[0][:200]
    f, t, S = spectrogram(data["X"][idx].mean(0), fs=FS, nperseg=64, noverlap=48)
    ax.pcolormesh(t, f, 10 * np.log10(S + 1e-10), shading="auto")
    ax.set_title(name)
axes[0].set_ylabel("Hz")
plt.tight_layout()

## Augmentation temporelle

In [ ]:
x = data["X"][np.where(data["y"] == CLASS_NAMES.index("V"))[0][0]]
rng = np.random.default_rng(1)
fig, axes = plt.subplots(1, 3, figsize=(12, 2.6), sharey=True)
for ax, (name, fn) in zip(axes, [("jittering", jitter), ("time warping", time_warp), ("magnitude warping", magnitude_warp)]):
    ax.plot(x, color="gray", lw=1, label="original")
    ax.plot(fn(x, rng), color="tab:red", lw=1, label=name)
    ax.set_title(name)
plt.tight_layout()

## Prédiction sur un enregistrement de test, avec explication

Le modèle est chargé depuis `results/<tag>/best.pt`. Les battements sont parcourus dans l'ordre temporel comme dans un flux.

In [ ]:
CKPT = ROOT / "results" / "resnet_aug" / "best.pt"   # adapter : cnn, resnet, cnn_lstm, ...
model = load_checkpoint(CKPT)
cam_engine = GradCAM1D(model) if hasattr(model, "features") else None

test_idx = splits["test"]
record = np.unique(data["rec"][test_idx])[0]
stream = np.where(data["rec"] == record)[0][:300]   # 300 premiers battements de l'enregistrement
wrong_or_rare = [i for i in stream if data["y"][i] != 0][:4] or list(stream[:4])

for i in wrong_or_rare:
    x = data["X"][i]
    cam, pred, _ = cam_engine(x)
    plot_gradcam(x, cam[0], title=f"enreg. {record} | réel {CLASS_NAMES[data['y'][i]]} | prédit {CLASS_NAMES[pred[0]]}")
    plt.show()

In [ ]:
# Attention (CNN-LSTM) : nécessite un checkpoint cnn_lstm
att_ckpt = ROOT / "results" / "cnn_lstm_aug" / "best.pt"
if att_ckpt.exists():
    att_model = load_checkpoint(att_ckpt)
    alpha, pred = attention_weights(att_model, data["X"][wrong_or_rare[0]])
    plot_attention(data["X"][wrong_or_rare[0]], alpha, pred)